# 🤖 Building AI Agents with LM Studio: From One Decision to an Autonomous Loop

### Dinesh AI Academy | Day 4 — Agents & MCP

**Learning objective:**
By the end of this notebook you will be able to explain, and build from scratch,
a working AI agent — no framework, no magic, just a local LM Studio model + a Python loop.

**Where we left off (Day 3):** we built one round trip — ask the model, it optionally
requests **one** tool, we run it, we send the result back, the model answers. That
round trip only ever makes **one decision**.

**Where we're going today:** what if the goal needs *several* decisions in a row,
and we don't know in advance how many, or in what order? That's an **agent**.

> ⚠️ **This notebook runs entirely against your local LM Studio server — no API key,
> no cloud, no cost.** Because it talks to `localhost`, it must be run **locally**
> (e.g. in VS Code), not in Google Colab. It reuses the same connection pattern as
> the Day 3 `4- Tools_LM_Studio.ipynb` notebook — if anything below is unfamiliar,
> that notebook covers the basics of tool calling against LM Studio in more depth.

## 1. What Is an Agent, Exactly?

> **An agent is a loop.** At every turn, the LLM looks at the goal and everything
> that has happened so far, and decides the *next* action — call a tool, or stop
> and answer. Your application keeps that loop running until the LLM decides
> it's done (or a safety limit is hit).

Compare the two mental models:

```text
Day 3 — a single tool-calling round trip (one decision):

   User question
        |
   LM Studio model -- decides once --> tool call OR final answer
        |
   (if tool)  run it, send result back, model answers
        |
      Done.


Day 4 — an agent (many decisions, repeated until done):

        +----------------------------------------+
        |                                         |
   Goal -> Model decides: tool, or final answer?   |
        |        |                     |           |
        |     tool call            final answer    |
        |        |                     |           |
        |   run the tool               v           |
        |        |                   Done.         |
        |   observe result                         |
        |        |                                 |
        +--------+   (loop back to "Model decides" again)
```

The only structural difference between Day 3 and today is: **we wrap the round
trip in a loop**, and keep feeding every tool result back in, so the model can
decide to call *another* tool based on what it just learned — or stop.

| Term | Definition | Who decides the next step? |
|---|---|---|
| **Tool** | A single capability (calculator, weather lookup, database query) | — |
| **Workflow** | A fixed sequence of steps *you* wrote in code | The developer, in advance |
| **Agent** | A loop where the model picks the next action based on the goal + what happened so far | The model, at runtime |

**One sentence to remember:** *Tool = capability. Workflow = a script. Agent = a script that lets the model choose its own next line.*

## 2. Setup — Connect to LM Studio

Unlike Gemini, there's no API key here — LM Studio runs an **OpenAI-compatible**
server on your own machine, and the official `openai` Python SDK can talk to it
directly by pointing `base_url` at `localhost` instead of OpenAI's servers.

1. Open LM Studio and make sure a **tool-calling capable** chat model is downloaded
   (Llama 3.1+, Qwen, and Gemma 3/4 instruct families generally support it).
2. Go to the **Developer** tab and click **Start Server** (default `http://localhost:1234`).
3. Run the cells below to list what's loaded, then set `CHAT_MODEL` to match exactly.

> **No rate limits, no retries needed:** since everything runs locally, there's no
> `RESOURCE_EXHAUSTED`-style quota to worry about like on Gemini's free tier —
> the only failure mode is the server not being started yet.

In [ ]:
# Install the OpenAI Python SDK -- LM Studio speaks the same API, so this is
# the only client library we need (no google-genai here).
%pip install -q openai

In [ ]:
from openai import OpenAI

LM_STUDIO_BASE_URL = "http://localhost:1234/v1"

# LM Studio doesn't check the API key, but the SDK requires some string.
client = OpenAI(base_url=LM_STUDIO_BASE_URL, api_key="lm-studio")

print("Models available in LM Studio:")
for m in client.models.list().data:
    print(" -", m.id)

In [ ]:
# Set this to exactly one of the model ids printed above --
# it must be a tool-calling capable model (see Section 2).
MODEL = "gemma-4-e2b-it-qat"

print("LM Studio client is ready.")
print("Model:", MODEL)

## 3. Give the Agent Some Tools

Three plain Python functions — nothing LM Studio-specific about them. Two are
carried over from Day 3 for continuity; `get_weather` is new and is what will
let us build a genuine **multi-step** goal further down (a question whose answer
needs the output of one tool as the *input* to another).

`get_weather` is intentionally **simulated** (a small fixed lookup table, no
internet call) — same idea as the Day 3 Streamlit demo: free, offline, and
100% reproducible for a classroom, with zero API keys or servers beyond your
own LM Studio instance.

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

def calculate(a: float, b: float, operation: str) -> float:
    """Perform a basic arithmetic calculation."""
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Cannot divide by zero.")
        return a / b
    else:
        raise ValueError(f"Unsupported operation: {operation}")


def get_current_time(timezone: str) -> dict:
    """Get the current date and time for an IANA timezone such as Asia/Tokyo."""
    now = datetime.now(ZoneInfo(timezone))
    return {
        "timezone": timezone,
        "date": now.strftime("%Y-%m-%d"),
        "time": now.strftime("%H:%M:%S"),
        "formatted": now.strftime("%A, %d %B %Y at %I:%M:%S %p")
    }


# A tiny, fixed, offline weather lookup -- simulated on purpose (see note above).
WEATHER_DB = {
    "tokyo": {"temp_c": 26, "condition": "sunny"},
    "paris": {"temp_c": 18, "condition": "cloudy"},
    "mumbai": {"temp_c": 31, "condition": "humid, partly cloudy"},
    "new york": {"temp_c": 21, "condition": "rainy"},
    "london": {"temp_c": 16, "condition": "overcast"},
}

def get_weather(city: str) -> dict:
    """Get the current simulated weather for a city (demo data, not a live API)."""
    data = WEATHER_DB.get(city.strip().lower())
    if data is None:
        return {"city": city, "temp_c": 20, "condition": "unknown", "note": "city not in demo dataset"}
    return {"city": city, **data}


# Quick sanity check -- call each function directly, no LM Studio involved yet.
print("Calculator:", calculate(25, 40, "multiply"))
print("Current time:", get_current_time("Asia/Kolkata"))
print("Weather:", get_weather("Tokyo"))

## 4. Describe the Tools to LM Studio

Same idea as Day 3: the model can only see this JSON menu, never your Python
source. LM Studio uses the standard **OpenAI tool-calling schema** —
`{"type": "function", "function": {name, description, parameters}}` — which is
a different shape from Gemini's `FunctionDeclaration`, but describes exactly
the same information. We also add a `TOOLBOX` dict mapping each tool's name to
the *actual* function — that's how our own code will turn "the model asked for
`get_weather`" back into a real function call in a moment.

In [ ]:
calculator_tool = {
    "type": "function",
    "function": {
        "name": "calculate",
        "description": "Performs basic arithmetic calculations.",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {"type": "number", "description": "The first number."},
                "b": {"type": "number", "description": "The second number."},
                "operation": {
                    "type": "string",
                    "description": "The arithmetic operation.",
                    "enum": ["add", "subtract", "multiply", "divide"]
                }
            },
            "required": ["a", "b", "operation"]
        }
    }
}

time_tool = {
    "type": "function",
    "function": {
        "name": "get_current_time",
        "description": "Gets the current date and time for an IANA timezone such as Asia/Kolkata or Asia/Tokyo.",
        "parameters": {
            "type": "object",
            "properties": {
                "timezone": {
                    "type": "string",
                    "description": "IANA timezone name, for example Asia/Kolkata, Asia/Tokyo, or America/New_York."
                }
            },
            "required": ["timezone"]
        }
    }
}

weather_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Gets the current simulated weather (temperature in Celsius, condition) for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name, e.g. 'Tokyo'."}
            },
            "required": ["city"]
        }
    }
}

# Maps a tool NAME (a string the model sends us) to the REAL function that runs it.
# This is the one place our code bridges "what the model asked for" to
# "what actually executes" -- see Section 6.
TOOLBOX = {
    "calculate": calculate,
    "get_current_time": get_current_time,
    "get_weather": get_weather,
}

TOOL_SCHEMAS = [calculator_tool, time_tool, weather_tool]

SYSTEM_PROMPT = (
    "You are a helpful assistant that can use tools to answer questions "
    "accurately. Use get_weather for weather questions, get_current_time "
    "for current time/date questions, and calculate for arithmetic where "
    "accuracy matters. If a question needs the result of one tool as input "
    "to another (for example, converting a temperature you just looked up), "
    "call the tools one after another rather than guessing the second value. "
    "If no tool is needed, answer directly."
)

print("Three tools are available to the model:", list(TOOLBOX.keys()))

# 🎯 5. Why a Single Round Trip Isn't Enough

Try a goal that needs **two tools, where the second one depends on the first's result**:

> "What's the current temperature in Tokyo in Celsius, and what is that in Fahrenheit?"

The model can't answer the Fahrenheit part without first knowing the Celsius value --
and it doesn't know that until `get_weather` actually runs. Watch what happens
with only **one** round trip (ask -> maybe one tool -> answer), exactly like Day 3:

In [ ]:
user_prompt = "What's the current temperature in Tokyo in Celsius, and what is that in Fahrenheit?"

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ],
    tools=TOOL_SCHEMAS,
    temperature=0,
)

message = response.choices[0].message
if message.tool_calls:
    for tc in message.tool_calls:
        print("Model requested:", tc.function.name, tc.function.arguments)
if message.content:
    print("Model text:", message.content)

Notice: the model can only request **one step at a time**. After this single
round trip, it has asked for the weather -- but it hasn't converted anything to
Fahrenheit yet, because it doesn't have the Celsius number in front of it yet.

If we stop here (like Day 3 did), the conversation is unfinished. We'd have to
manually: run the tool, send the result back, check if the model asks for
*another* tool, run that too, send it back again... That's exactly the
repetition a loop is for.

## 6. Build the Agent Loop

This is the entire idea of an agent, in code. Compare it to the diagram in
Section 1 -- it's the *same* four steps (ask -> decide -> act -> observe), just
wrapped in a loop that keeps going until the model stops asking for tools:

```text
for step in range(max_steps):
    1. Ask the model, with the full conversation so far
    2. If the model's reply has NO tool_calls -> done, return the text
    3. Otherwise: run the requested tool(s) ourselves
    4. Append the tool result(s) to the conversation (role="tool")
    5. Loop back to step 1 -- the model now sees the result and decides again
```

A `max_steps` safety cap exists because a model can occasionally get stuck
re-requesting the same tool instead of answering -- we never want an agent that
can loop forever (or, on a slower local machine, burn several minutes of
compute for no reason).

In [ ]:
import json

def execute_tool(name: str, args: dict):
    """Turn a tool name + arguments (from the model) into a real function call."""
    func = TOOLBOX.get(name)
    if func is None:
        raise ValueError(f"Unknown tool requested: {name}")
    return func(**args)


def call_model(messages, retries: int = 3, backoff_seconds: float = 3.0):
    """
    chat.completions.create() with a tiny retry loop -- not for rate limits
    (there are none locally), but for the occasional connection hiccup if the
    LM Studio server is still loading the model into memory.
    """
    import time
    for attempt in range(1, retries + 1):
        try:
            return client.chat.completions.create(
                model=MODEL, messages=messages, tools=TOOL_SCHEMAS, temperature=0,
            )
        except Exception as e:
            if attempt < retries:
                print(f"Request failed ({e}); retrying in {backoff_seconds:.0f}s "
                      f"({attempt}/{retries - 1})... is the LM Studio server running?")
                time.sleep(backoff_seconds)
            else:
                raise


def run_agent(goal: str, max_steps: int = 5, verbose: bool = True) -> str:
    """
    The agent loop: repeatedly ask the model, execute whatever tool it requests,
    and feed the result back -- until it answers with plain text, or we hit
    max_steps.
    """
    # `messages` is the agent's growing memory of this run: the system prompt,
    # the goal, every tool-call request the model made, and every tool result
    # we sent back.
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": goal},
    ]

    if verbose:
        print(f"GOAL: {goal}\n{'-' * 60}")

    for step in range(1, max_steps + 1):
        response = call_model(messages)
        model_message = response.choices[0].message

        # Keep the model's own turn in the conversation before we look at it,
        # so the next request includes exactly what it said this time.
        messages.append(model_message)

        if not model_message.tool_calls:
            # No tool requested this turn -- the model considers itself done.
            final_text = model_message.content or ""
            if verbose:
                print(f"Step {step}: model answered directly (no tool needed).")
                print(f"\nFINAL ANSWER:\n{final_text}")
            return final_text

        # Run every tool the model asked for this turn (it can request more
        # than one tool call in a single turn).
        for tool_call in model_message.tool_calls:
            args = json.loads(tool_call.function.arguments or "{}")
            if verbose:
                print(f"Step {step}: model requested tool `{tool_call.function.name}` with {args}")
            try:
                result = execute_tool(tool_call.function.name, args)
            except Exception as e:
                result = {"error": str(e)}
            if verbose:
                print(f"         -> result: {result}")
            response_payload = result if isinstance(result, dict) else {"result": result}
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(response_payload),
            })

    # Safety net: the loop ran out of steps without the model producing a final answer.
    if verbose:
        print(f"Stopped after {max_steps} steps without a final answer.")
    return "(agent hit the step limit before finishing)"

print("Agent loop is ready.")

## 7. Run It — the Two-Tool, Chained Goal

Same question as Section 5, but this time through `run_agent()`. Watch the
printed trace: the model should call `get_weather` first, see the Celsius value,
*then* call `calculate` to convert it -- two decisions, made one after another,
each informed by what came before.

In [ ]:
run_agent("What's the current temperature in Tokyo in Celsius, and what is that in Fahrenheit?")

## 8. Run It — a Few More Goals

Try goals that need a different number of steps, so you can see the loop
adapt: zero tools, one tool, and two *independent* tools in one go.

In [ ]:
# No tool needed at all -- the model should answer directly on step 1.
run_agent("In one sentence, what is an AI agent?")

In [ ]:
# Exactly one tool.
run_agent("What time is it right now in Tokyo?")

In [ ]:
# Two tools, but independent of each other (order doesn't matter, unlike Section 7).
run_agent("What's the weather in Paris, and what time is it there right now?")

# 🧠 9. What Actually Makes This an *Agent* (and Not Just a Longer Workflow)?

Look closely at `run_agent()`: notice what our Python code does **not** decide.

- We never wrote `if "weather" in goal: call get_weather()`.
- We never hardcoded "always call the weather tool before the calculator."
- We never told it how many steps a particular goal would take.

**The model decided all of that, at runtime, from the goal text alone** -- which
tool(s) to call, in what order, whether a second tool was even needed, and
when to stop. Our code only supplied the *capabilities* (the tools) and the
*loop* (keep going until done). That division of responsibility is the whole
definition of an agent:

| | Workflow | Agent |
|---|---|---|
| Who picks the next step? | The developer, hardcoded in advance | The model, at runtime |
| Can it skip a step it doesn't need? | No -- the code always runs it | Yes -- it just won't call that tool |
| Can it handle a goal you didn't anticipate? | Only if you wrote a branch for it | Often yes, if the right tools exist |
| Predictability | High | Lower -- needs guardrails (next section) |

# 🛡️ 10. Production Safety Note — Agents Need *More* Guardrails Than a Single Tool Call

An agent that can take multiple, self-chosen steps is more powerful -- and more
dangerous -- than a single tool call. Everything from Day 3's safety note still
applies, plus:

- **`max_steps` (we used 5)** -- without it, a confused model can loop
  indefinitely. Locally this costs *time and GPU/CPU cycles* instead of API
  dollars, but an unbounded loop is still a real problem on a shared machine.
- **Latency awareness** -- every step in the loop is a *full* inference call.
  A 5-step agent run on a local model can take noticeably longer than a single
  request, especially on modest hardware. Log step counts in production.
- **Tool allow-lists** -- `TOOLBOX` only contains what we explicitly added.
  Never let a model call an arbitrary function by name.
- **Idempotency for risky tools** -- if a tool sends an email or charges a card,
  a retried step could run it twice. Read-only tools like ours are safe to
  repeat; side-effecting ones need extra care.
- **Human approval before high-impact actions** -- exactly like Day 3: the loop
  can *request* an action, but your application should still gate anything
  irreversible (payments, deletions, sending messages) behind a real check.
- **Timeouts per step** -- a single slow tool call (e.g. a hanging network
  request) can stall the entire agent. Add per-tool timeouts in real systems.

```text
Agent requests: send_email(to="someone@example.com")
                              |
                 Your application checks:
                 Is this user allowed? Is this within the step budget?
                 Does policy require human approval first?
                              |
                     Execute / Reject
```

# 🧪 11. Classroom Challenge

For each goal below, predict **before running it**: how many tool calls will
the agent make, and in what order? Then try it and compare.

| Goal | Your prediction |
|---|---|
| "Convert 45°C to Fahrenheit." | ? |
| "What's 18% of 2400, and what time is it in London?" | ? |
| "Is it warmer in Mumbai or Paris right now?" | ? |
| "What is the capital of France?" | ? |

The last one is a trick question for a reason -- **no tool we built can answer
"capital of France," and none is needed.** A well-built agent recognizes that
too, and just answers from its own knowledge instead of forcing a tool call.

> Not every local model handles this equally well -- smaller models are more
> prone to calling a tool "just in case" even when the answer needs no tool at
> all. If you see that happen, it's a good, concrete illustration of *why*
> tool-calling reliability varies by model, and why the system prompt's last
> line ("If no tool is needed, answer directly.") matters.

In [ ]:
run_agent("Is it warmer in Mumbai or Paris right now?")

## 🌉 Next: MCP — Reusable Tools You Didn't Have to Build

Every tool in this notebook was something **we** wrote by hand: the Python
function *and* its JSON schema description. That's fine for three toy tools --
but real agents often need access to dozens of tools built by other teams
entirely (a company database, a filesystem, Slack, GitHub, a CRM), and
hand-writing + maintaining a schema for each one doesn't scale.

**MCP (Model Context Protocol)** is a standard that lets an agent *discover*
and *call* tools exposed by any MCP-compatible server, without you writing
custom integration code for each one -- the server describes its own tools in
a standard format, and any MCP-aware agent (built with any SDK, in any
language, talking to any model -- cloud or local) can use them immediately.

That's the next notebook: connecting the agent loop you just built to tools
you *didn't* have to write yourself.

# 🎓 Day 4 Takeaway

By the end of this notebook, you should be able to explain:

1. What an agent is, in one sentence, and how it differs from a single
   tool-calling round trip and from a hardcoded workflow.
2. Why a fixed number of tool calls isn't enough for goals whose steps
   depend on each other.
3. The five parts of the agent loop: ask -> decide -> act -> observe -> repeat.
4. Why `max_steps` (or an equivalent safety cap) is not optional in a real
   agent.
5. What makes something an *agent* rather than a workflow: **who** decides
   the next step, and **when**.
6. Why agents need stronger safety guardrails than a single tool call.
7. That the agent loop itself is **provider-agnostic** -- the exact same
   five steps ran here against a local LM Studio model as they did against
   Gemini earlier; only the request/response shapes (`tools=` schema,
   `tool_calls` vs `function_call`, `role="tool"` messages) changed.

### The complete mental model

```text
        Goal
          |
   +------v------+
   | LM Studio    |<----------------+
   | model        |                 |
   |  (decide)    |                 |
   +------+------+                 |
          |                        |
   tool needed?  -- no --> Final answer
          | yes                    |
   +------v------+                 |
   | Run the real |                |
   | Python tool  |                |
   +------+------+                 |
          |                        |
     Observe result ---------------+
     (feed back in, loop again)
```

## Official references

- LM Studio — Local server & OpenAI compatibility: https://lmstudio.ai/docs/app/api
- LM Studio — Tool use / function calling: https://lmstudio.ai/docs/app/api/tools
- OpenAI API — Function calling: https://platform.openai.com/docs/guides/function-calling
- OpenAI Python SDK: https://github.com/openai/openai-python
- Model Context Protocol (MCP) — Introduction: https://modelcontextprotocol.io/introduction

This notebook intentionally builds the agent loop manually (no agent framework)
so every decision point is visible. Frameworks like LangChain/LangGraph, the
OpenAI Agents SDK, and Google's ADK automate exactly this loop -- now that
you've built one by hand, against both a cloud model (Gemini) and a local one
(LM Studio), you'll recognize the same five steps inside any of them, regardless
of which model is actually answering.